In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
from math import sqrt
import ROOT
import ctypes
try:
#     plt.style.use('belle2')
    plt.style.use('belle2_serif')
#     plt.style.use('belle2_modern')
except OSError:
    print("Please install belle2 matplotlib style") 
px = 1/plt.rcParams['figure.dpi']

from main.data_tools.extract_ntuples import get_pd, get_np
from main.draw_tools.decorations import b2helix, watermark
from main.draw_tools.stacking_with_error_bars import MC_stack_plot, MC_stack_plot_density

from main.data_tools.error_bars import make_data_weight
from main.data_tools.query_dataframes import cut_dfs_7types

from matplotlib.ticker import ScalarFormatter


Welcome to JupyROOT 6.26/04


In [2]:
def cal_Br_Central_value(Nsig, Nref, eff_sig, eff_ref, Br_ref_dec):
    print("Input values:")
    print("Nsig:", Nsig)
    print("Nref:", Nref)
    print("eff_sig:", eff_sig)
    print("eff_ref:", eff_ref)
    print("Br_ref_dec:", Br_ref_dec)
    
    N_real_sig = Nsig / eff_sig
    N_real_ref = Nref / eff_ref
    
    Br = Br_ref_dec * (N_real_sig / N_real_ref)
    
    return Br
    
from math import sqrt

def cal_Br_error_stat(Nsig_err, Nsig, Nref_err, Nref, central_value):
    Variance = (Nsig_err / Nsig)**2 + (Nref_err / Nref)**2
    TOTAL = sqrt(Variance) * central_value
    return TOTAL

def cal_Br_error_with_eff(Nsig_err, Nsig, Nref_err, Nref, eff_sig_err, eff_sig, eff_ref_err, eff_ref, central_value):
    Variance = (Nsig_err / Nsig)**2 + (eff_sig_err / eff_sig)**2 + (Nref_err / Nref)**2 + (eff_ref_err / eff_ref)**2
    TOTAL = sqrt(Variance) * central_value
    return TOTAL


In [3]:
from math import sqrt

# Function to calculate the central branching ratio value
def calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_dec):
    N_real_sig = Nsig / eff_sig
    N_real_ref = Nref / eff_ref
    Br = Br_ref_dec * (N_real_sig / N_real_ref)
    return Br
    
def calculate_br_ratio(Nsig,Nsig_err,  Nref,Nref_err, eff_sig, eff_ref):
    N_real_sig = Nsig / eff_sig
    N_real_ref = Nref / eff_ref
    Br_ratio = (N_real_sig / N_real_ref)
    Br_ratio_err = (eff_ref/eff_sig) * math.sqrt( (Nsig_err / Nref)**2 + (Nsig / (Nref**2) * Nref_err)**2 )
    print(f'Br_ratio value = {Br_ratio:.4e}')
    print(f'Br_ratio Statistical uncertainty = {Br_ratio_err:.4e}')
    return Br_ratio, Br_ratio_err

# Function to calculate statistical uncertainty
def calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value):
    variance = (Nsig_err / Nsig)**2 + (Nref_err / Nref)**2
    return sqrt(variance) * central_value

# Function to print results in a formatted way
def print_results(label, central_value, stat_unc, Br_sig_dec):
    pull = (central_value - Br_sig_dec) / stat_unc
    print(f'{label} Central value = {central_value:.4e}')
    print(f'{label} Statistical uncertainty = {stat_unc:.4e}')
    print(f'{label} Pull = {pull:.4f}')
    print(f'{label} stas. unc./Central value = {stat_unc/central_value:.4e}\n')

# Error-weighted combination function
def combine_error_weighted(x, y, x_err, y_err):
    central_value = (x / x_err**2 + y / y_err**2) / (1 / x_err**2 + 1 / y_err**2)
    error = 1 / sqrt(1 / x_err**2 + 1 / y_err**2)
    return central_value, error

In [4]:
# signal_eff_error = math.sqrt(signal_eff * (1 - signal_eff) / N_gen)

def calculate_sig_eff_err(eff, N_gen):

    error = math.sqrt(eff * (1 - eff) / N_gen)
    return error

# Br(D+ -> eta K+)

In [20]:
# Constants
Br_ref_dec = 0.003770000
Br_sig_dec = 0.000125000

Br_sig_PDG = 0.0001250000
Br_sig_PDG_err = 0.0000160000


In [21]:
# 0.91
eff_sig_cal =  0.0455
eff_ref_cal =  0.0631

In [22]:
eff_sig_err_cal = calculate_sig_eff_err(eff_sig_cal, 6e+6)

eff_sig_err_cal


eff_ref_err_cal = calculate_sig_eff_err(eff_ref_cal, 6e+6)
eff_ref_err_cal

print(f"signal eff error: {eff_sig_err_cal:.6e}, ref eff error: {eff_ref_err_cal:.6e}")

signal eff error: 8.507815e-05, ref eff error: 9.926261e-05


In [23]:
# First calculation for mode: eta -> gg
Nsig_err, Nsig, Nref_err, Nref = 86.5031577561831, 2135.3341387235146 , 420.2624517243603  , 89632.29889637447
# Nsig_err, Nsig, Nref_err, Nref = , , ,

eff_sig_err, eff_sig, eff_ref_err, eff_ref = eff_sig_err_cal, eff_sig_cal, eff_ref_err_cal, eff_ref_cal

central_value_1 = calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_dec)
stat_unc_1 = calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value_1)
print_results("Mode: eta -> gg", central_value_1, stat_unc_1, Br_sig_dec)


Mode: eta -> gg Central value = 1.2455e-04
Mode: eta -> gg Statistical uncertainty = 5.0794e-06
Mode: eta -> gg Pull = -0.0876
Mode: eta -> gg stas. unc./Central value = 4.0781e-02



In [24]:
ratio_central_Br_gg, ratio_err_Br_gg  = calculate_br_ratio(Nsig,Nsig_err, Nref, Nref_err, eff_sig, eff_ref)


Br_ratio value = 3.3038e-02
Br_ratio Statistical uncertainty = 1.3473e-03


In [25]:
# 0.92
eff_sig_cal =  0.0381
eff_ref_cal = 0.0524

In [26]:
eff_sig_err_cal = calculate_sig_eff_err(eff_sig_cal, 6e+6)

eff_ref_err_cal = calculate_sig_eff_err(eff_ref_cal, 6e+6)

print(f"signal eff error: {eff_sig_err_cal:.6e}, ref eff error: {eff_ref_err_cal:.6e}")

signal eff error: 7.815411e-05, ref eff error: 9.097091e-05


In [27]:
# Second calculation for mode: eta -> pipipi
Nsig_err, Nsig, Nref_err, Nref = 46.589920081782736,  1059.7601523904102, 247.2642529957775,45094.21598660429
# Nsig_err, Nsig, Nref_err, Nref =, , ,

eff_sig_err, eff_sig, eff_ref_err, eff_ref = eff_sig_err_cal, eff_sig_cal ,eff_ref_err_cal,eff_ref_cal

central_value_2 = calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_dec)
stat_unc_2 = calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value_2)
print_results("Mode: eta -> pipipi", central_value_2, stat_unc_2, Br_sig_dec)

Mode: eta -> pipipi Central value = 1.2185e-04
Mode: eta -> pipipi Statistical uncertainty = 5.3985e-06
Mode: eta -> pipipi Pull = -0.5830
Mode: eta -> pipipi stas. unc./Central value = 4.4303e-02



In [28]:
ratio_central_Br_3pi, ratio_err_Br_3pi  = calculate_br_ratio(Nsig,Nsig_err, Nref, Nref_err, eff_sig, eff_ref)


Br_ratio value = 3.2322e-02
Br_ratio Statistical uncertainty = 1.4320e-03


In [29]:
# Combined error-weighted result of absoulte Br
combined_central_value, combined_error = combine_error_weighted(central_value_1, central_value_2, stat_unc_1, stat_unc_2)
print_results("Combined", combined_central_value, combined_error, Br_sig_dec)

Combined Central value = 1.2329e-04
Combined Statistical uncertainty = 3.6994e-06
Combined Pull = -0.4634
Combined stas. unc./Central value = 3.0006e-02



In [30]:
# Combined error-weighted result of Br ratio
combined_central_value, combined_error = combine_error_weighted(ratio_central_Br_gg,ratio_central_Br_3pi, ratio_err_Br_gg, ratio_err_Br_3pi)
print_results("Combined", combined_central_value, combined_error, 0.125/3.77)

0.125/3.77

Combined Central value = 3.2702e-02
Combined Statistical uncertainty = 9.8126e-04
Combined Pull = -0.4634
Combined stas. unc./Central value = 3.0006e-02



0.033156498673740056

In [31]:
# # Plotting
# plt.errorbar(1, central_value_1, yerr=stat_unc_1, fmt='o', capsize=4,color='blue')
# plt.errorbar(2, central_value_2, yerr=stat_unc_2, fmt='o', capsize=4, color='blue')
# plt.errorbar(3, combined_central_value, yerr=combined_error, fmt='o', capsize=4, color='red')
# plt.errorbar(4, Br_sig_dec, yerr=0, fmt='o', capsize=4, label='Input',color='gray')
# plt.errorbar(5, Br_sig_PDG, yerr=Br_sig_PDG_err, fmt='o', capsize=4,color='gray')
# plt.errorbar(6, 0.151/1000, yerr=0.025/1000, fmt='o', capsize=4,color='gray')
# plt.errorbar(7, 1.08/10000, yerr=0.17/10000, fmt='o', capsize=4,color='gray')

# # Customizing plot
# plt.xticks([1, 2, 3, 4, 5, 6, 7], [r'$D^+ \to \eta_{\gamma\gamma} K^+$', r'$D^+ \to \eta_{3\pi} K^+$', 'Combined','Input','PDG','BESIII\n(2018)','Belle\n(2011)'], fontsize=11)
# # plt.xticks([1, 2, 3, 4], [r'$D^+ \to \eta_{\gamma\gamma} K^+$', r'$D^+ \to \eta_{3\pi} K^+$', 'Combined','Input','PDG'], fontsize=12)

# plt.gca().yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
# plt.ticklabel_format(axis="y", style="sci", scilimits=(0,0))

# plt.ylabel('Branching fraction')
# plt.grid(True, alpha=0.5)
# plt.title("Only stats. uncertainty")
# # plt.legend()
# plt.tight_layout()
# plt.savefig("MC15rd_427_87_etaKp_gg_Br_fix_v10_bdt_Dp_CMS_p_new_Ds_correct.best_Nsigma.weighted.png")
# # plt.savefig("MC15rd_427_87_etaKp_gg_Br_fix_v7_bdt_Dp_CMS_p_new_range.best_FOM.weighted.png")

# plt.show()

# Br(Ds+ -> eta K+)

In [32]:
# Constants
Br_ref_dec = 0.017000000
Br_sig_dec = 0.001600000

Br_sig_PDG = 0.001730000
Br_sig_PDG_err = 0.000080000


In [33]:
# 0.91
eff_sig_cal = 0.0386
eff_ref_cal = 0.0544

In [34]:
eff_sig_err_cal = calculate_sig_eff_err(eff_sig_cal,  6e+6)
eff_ref_err_cal = calculate_sig_eff_err(eff_ref_cal, 6e+6)
print(f"signal eff error: {eff_sig_err_cal:.6e}, ref eff error: {eff_ref_err_cal:.6e}")

signal eff error: 7.864481e-05, ref eff error: 9.259287e-05


In [35]:
# First calculation for mode: eta -> gg
Nsig_err, Nsig, Nref_err, Nref = 146.61129160463406 , 12508.210298986762 , 484.9344103310723,189091.01335422136
# Nsig_err, Nsig, Nref_err, Nref =  , , ,

eff_sig_err, eff_sig, eff_ref_err, eff_ref = eff_sig_err_cal, eff_sig_cal, eff_ref_err_cal, eff_ref_cal

central_value_1 = calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_dec)
stat_unc_1 = calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value_1)
print_results("Mode: eta -> gg", central_value_1, stat_unc_1, Br_sig_dec)

Mode: eta -> gg Central value = 1.5848e-03
Mode: eta -> gg Statistical uncertainty = 1.9016e-05
Mode: eta -> gg Pull = -0.7974
Mode: eta -> gg stas. unc./Central value = 1.1998e-02



In [36]:
ratio_central_Br_gg, ratio_err_Br_gg  = calculate_br_ratio(Nsig,Nsig_err, Nref, Nref_err, eff_sig, eff_ref)


Br_ratio value = 9.3226e-02
Br_ratio Statistical uncertainty = 1.1186e-03


In [37]:
# 0.92
eff_sig_cal  =  0.0276
eff_ref_cal =  0.0392

In [38]:
eff_sig_err_cal = calculate_sig_eff_err(eff_sig_cal, 6e+6)
eff_sig_err_cal
# eff_ref_err_cal = calculate_sig_eff_err(0.02653, 2e+6)
eff_ref_err_cal = calculate_sig_eff_err(eff_ref_cal, 6e+6)

print(f"signal eff error: {eff_sig_err_cal:.6e}, ref eff error: {eff_ref_err_cal:.6e}")

signal eff error: 6.688079e-05, ref eff error: 7.922895e-05


In [39]:
# Second calculation for mode: eta -> pipipi
Nsig_err, Nsig, Nref_err, Nref = 82.14332050154917, 5242.557756749183, 298.0571484768516, 79195.03799931434
# Nsig_err, Nsig, Nref_err, Nref = , , ,
eff_sig_err, eff_sig, eff_ref_err, eff_ref = eff_sig_err_cal, eff_sig_cal , eff_ref_err_cal, eff_ref_cal

central_value_2 = calculate_br(Nsig, Nref, eff_sig, eff_ref, Br_ref_dec)
stat_unc_2 = calculate_stat_uncertainty(Nsig_err, Nsig, Nref_err, Nref, central_value_2)
print_results("Mode: eta -> pipipi", central_value_2, stat_unc_2, Br_sig_dec)

Mode: eta -> pipipi Central value = 1.5983e-03
Mode: eta -> pipipi Statistical uncertainty = 2.5756e-05
Mode: eta -> pipipi Pull = -0.0642
Mode: eta -> pipipi stas. unc./Central value = 1.6114e-02



In [40]:
ratio_central_Br_3pi, ratio_err_Br_3pi  = calculate_br_ratio(Nsig,Nsig_err, Nref, Nref_err, eff_sig, eff_ref)


Br_ratio value = 9.4020e-02
Br_ratio Statistical uncertainty = 1.5151e-03


In [41]:
# Combined error-weighted result
combined_central_value, combined_error = combine_error_weighted(central_value_1, central_value_2, stat_unc_1, stat_unc_2)
print_results("Combined", combined_central_value, combined_error, Br_sig_dec)

Combined Central value = 1.5896e-03
Combined Statistical uncertainty = 1.5298e-05
Combined Pull = -0.6796
Combined stas. unc./Central value = 9.6238e-03



In [42]:
# Combined error-weighted result
combined_central_value, combined_error = combine_error_weighted(ratio_central_Br_gg,ratio_central_Br_3pi, ratio_err_Br_gg, ratio_err_Br_3pi)
print_results("Combined", combined_central_value, combined_error, 0.160/1.70)

0.160/1.70

Combined Central value = 9.3506e-02
Combined Statistical uncertainty = 8.9988e-04
Combined Pull = -0.6796
Combined stas. unc./Central value = 9.6238e-03



0.09411764705882353

In [44]:
# # Plotting
# plt.errorbar(1, central_value_1, yerr=stat_unc_1, fmt='o', capsize=4,color='blue')
# plt.errorbar(2, central_value_2, yerr=stat_unc_2, fmt='o', capsize=4,color='blue')
# plt.errorbar(3, combined_central_value, yerr=combined_error, fmt='o', capsize=4,color='red')
# plt.errorbar(4, Br_sig_dec, yerr=0, fmt='o', capsize=4, label='Input',color='gray')
# plt.errorbar(5, Br_sig_PDG, yerr=Br_sig_PDG_err, fmt='o', capsize=4,color='gray')
# plt.errorbar(6, 1.75/1000, yerr=0.05/1000, fmt='o', capsize=4,color='gray')
# plt.errorbar(7, 1.62/1000, yerr=0.1/1000, fmt='o', capsize=4,color='gray')

# # Customizing plot
# plt.xticks([1, 2, 3, 4, 5, 6, 7], [r'$D_s^+ \to \eta_{\gamma\gamma} K^+$', r'$D_s^+ \to \eta_{3\pi} K^+$', 'Combined','Input','PDG','Belle\n(2021)','BESIII\n(2020)'], fontsize=12)
# # plt.xticks([1, 2, 3, 4], [r'$D^+ \to \eta_{\gamma\gamma} K^+$', r'$D^+ \to \eta_{3\pi} K^+$', 'Combined','Input','PDG'], fontsize=12)

# plt.gca().yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
# plt.ticklabel_format(axis="y", style="sci", scilimits=(0,0))

# plt.ylabel('Branching fraction')
# plt.grid(True, alpha=0.5)
# plt.title("Only stats. uncertainty")

# # plt.legend()
# plt.tight_layout()
# # plt.savefig("MC15rd_427_87_etaKp_gg_Ds_Br_v7.png")
# plt.savefig("MC15rd_427_87_etaKp_gg_Ds_Br_fix_v10_bdt_Dp_CMS_p_new_Ds_correct.best_Nsigma.weighted.png")
# # plt.savefig("MC15rd_427_87_etaKp_gg_Ds_Br_fix_v7_bdt_Dp_CMS_p_new_range.best_FOM.weighted.png")


# plt.show()